# Section 5.9 — An Illustrative Implementation of the Rolling SP

Demonstrates a full rolling-horizon SP simulation across a 36-month real
horizon. This is a self-contained illustration of the operational behaviour
of the SP-model framework when rolled iteratively through time.

**Setup**

| Setting | Value |
|---------|-------|
| Planning horizon per roll | 30 months |
| Commit window | 3 months |
| Number of rolls | 12 (covering 36 months) |
| SP structure | 3-month deterministic prefix + 3 stochastic stages × 9 months = 27 scenarios |
| MIP gap | 3% per sub-MILP |
| Instance | 60-cage regional fleet (6 locations) |
| Temperature paths | 5 named realisations |

**Expected runtime**: ~5–6 hours total across all 5 paths (≈ 60 PH solves).
Results are saved per roll so progress is preserved if interrupted.

**Dependencies** (all local to this folder):
- `ip.py` — per-scenario MILP subproblem
- `sp.py` — tree-aware B-PHA solver (27-scenario, 3-stage tree)
- `forward_sim.py` — forward simulator (3-month commit window)
- `instance.py` — 60-cage fleet, scenario tree definition, and temperature profiles
- `rolling_horizon.py` — top-level rolling-horizon driver


In [ ]:
import os, sys, time, pickle, json
import numpy as np
import pandas as pd

# ── Path setup ──────────────────────────────────────────────────────────────
# All dependencies live in this folder (ip.py, sp.py, forward_sim.py,
# instance.py, rolling_horizon.py). No external models/ directory needed.
HERE = os.path.dirname(os.path.abspath('__file__'))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

from scripts.instance import (
    units_df, loc_mab, regional_mab,
    temp_map, S_normal_v, S_bad_v,
    prefix_months, stage_slices,
)
from scripts.rolling_horizon import run_rolling_horizon, REALITY_PATHS

print('Imports OK')
print(f'  HERE = {HERE}')
print(f'  Units: {len(units_df)} rows, {units_df["location"].nunique()} locations')
print(f'  Regional MAB: {regional_mab:,} kg')
print(f'  S_normal: {S_normal_v:.6f}   S_bad: {S_bad_v:.6f}')
print(f'  Prefix months: {prefix_months}')
print(f'  Stage slices: {[f"[{sl[0]}..{sl[-1]}]" for sl in stage_slices]}')
print(f'\nAvailable reality paths: {list(REALITY_PATHS)}')


Imports OK
  HERE = c:\Users\Isak\OneDrive - University of Bergen\Dokumenter\Programmeringsfiler\master\fork_ulrik_models\experimental_evaluation\5.9_SP_simulation
  Units: 60 rows, 6 locations
  Regional MAB: 35,000,000 kg
  S_normal: 0.994017   S_bad: 0.970431
  Prefix months: [0, 1, 2]
  Stage slices: ['[3..11]', '[12..20]', '[21..29]']

Available reality paths: ['all_normal', 'warming', 'cooling', 'oscillating', 'stress_then_recover']


In [2]:
# ── Temperature realisations used in the experiment ────────────────────────
# Each path is a 12-step sequence of 3-month roll labels.
# Labels: 'normal', 'bad' (cold), 'good' (warm)

print('Reality paths (12 rolls × 3 months = 36 months):\n')
for name, path in REALITY_PATHS.items():
    abbrev = {'normal':'N', 'bad':'C', 'good':'W'}
    seq = ' '.join(abbrev.get(l,'?') for l in path)
    print(f'  {name:<22} {seq}')

print()
print('  N = normal temperatures')
print('  C = cold temperatures  (bad)')
print('  W = warm temperatures  (good)')


Reality paths (12 rolls × 3 months = 36 months):

  all_normal             N N N N N N N N N N N N
  warming                N N N W W N N W W N W W
  cooling                N N C C N N C C N C C C
  oscillating            N C W N C W N C W N C W
  stress_then_recover    C C C N N N W W W N N N

  N = normal temperatures
  C = cold temperatures  (bad)
  W = warm temperatures  (good)


In [3]:
# ── NPV computation from roll logs ─────────────────────────────────────────

ANNUAL_RATE           = 0.10
SMOLT_COST_PER_HEAD   = 10.0
TERMINAL_VALUE_PER_KG = 60.0

PRICE_BREAKS = [
    (1.0, 2.0, 39.72), (2.0, 3.0, 52.66), (3.0, 4.0, 60.73),
    (4.0, 5.0, 63.33), (5.0, 6.0, 64.55), (6.0, 7.0, 64.14),
    (7.0, 8.0, 62.85), (8.0, 9.0, 61.32), (9.0, 1e9, 58.80),
]

def price_for_weight_g(weight_g):
    w = weight_g / 1000.0
    for lo, hi, p in PRICE_BREAKS:
        if lo <= w < hi: return p
    return 58.80

def df(t_months): return (1 + ANNUAL_RATE) ** (-t_months / 12.0)

def compute_npv_from_rolls(roll_logs, units_df_final=None, real_horizon=36):
    """
    Compute discounted NPV from a list of per-roll logs (rolling_horizon.py format).
    Includes: harvest revenue, smolt cost, terminal value.
    Feed costs are excluded (tracked inside MILP objective but not in fwd_log).
    """
    npv = 0.0
    for k, log in enumerate(roll_logs):
        t_start = k * 3    # 3-month commit window per roll
        fwd_log = log.get('fwd_log', [])
        for ev in fwd_log:
            t   = t_start + int(ev.get('month', 0))
            d   = df(t)
            evt = ev.get('event', '')
            if evt in ('harvest_existing', 'harvest_new'):
                count  = float(ev.get('count_before', 0))
                wg     = float(ev.get('weight_g_before', 0))
                bio_kg = count * wg / 1000.0
                npv   += d * bio_kg * price_for_weight_g(wg)
            elif evt == 'stocked_new':
                q    = float(ev.get('q', 0))
                npv -= d * q * SMOLT_COST_PER_HEAD
    if units_df_final is not None:
        for _, row in units_df_final.iterrows():
            cnt = row.get('count'); wg = row.get('avg_weight_g')
            if cnt is not None and wg is not None:
                if not (pd.isna(cnt) or pd.isna(wg)):
                    bio_kg = float(cnt) * float(wg) / 1000.0
                    npv   += df(real_horizon) * TERMINAL_VALUE_PER_KG * bio_kg
    return npv

print('NPV helper defined.')


NPV helper defined.


## Run Rolling Horizon

Each path solves 12 consecutive SP problems (one per 3-month roll).
Each SP has 27 scenarios. Runtime per path is approximately 20–60 minutes
depending on PH convergence speed.

The `run_rolling_horizon` function saves per-roll results under `runs/<path_name>/`
and returns a dict with `roll_logs` and `config`. Already-completed paths are
skipped (the complete `result.pkl` is detected and reloaded).


In [ ]:
# ── Run all 5 paths ─────────────────────────────────────────────────────────
# Edit PATHS_TO_RUN to run a subset (e.g. ['all_normal'] for a quick smoke test).
# Each completed path is saved to runs/<path_name>/result.pkl.

SAVE_DIR    = os.path.join(HERE, 'runs')
PH_KWARGS   = {'K': 200, 'mip_gap': 0.03}
N_ROLLS     = 12        # 12 × 3 months = 36 months
PATHS_TO_RUN = list(REALITY_PATHS)   # run all 5; change to subset if desired
# PATHS_TO_RUN = ['all_normal']      # ← uncomment for a quick smoke test

os.makedirs(SAVE_DIR, exist_ok=True)
all_results = {}

print(f'Running rolling SP: {N_ROLLS} rolls × 3-month commit, T=30 months')
print(f'PH kwargs: {PH_KWARGS}')
print(f'Paths: {PATHS_TO_RUN}')
print(f'Saving to: {SAVE_DIR}\n')
t0_global = time.time()

for path_name in PATHS_TO_RUN:
    path = REALITY_PATHS[path_name][:N_ROLLS]
    path_dir = os.path.join(SAVE_DIR, path_name)
    pkl_path = os.path.join(path_dir, 'result.pkl')

    if os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            existing = pickle.load(f)
        if len(existing.get('roll_logs', [])) >= N_ROLLS:
            print(f'[SKIP] {path_name} — already completed ({len(existing["roll_logs"])} rolls found)')
            all_results[path_name] = existing
            continue

    print(f'\n>>> Running path: {path_name}  ({N_ROLLS} rolls)')
    t0_path = time.time()

    result = run_rolling_horizon(
        units_df0=units_df,
        loc_mab=loc_mab,
        regional_mab=regional_mab,
        reality_path=path,
        horizon_months=30,
        prefix_months=prefix_months,
        stage_slices=stage_slices,
        months_per_roll=3,
        start_calendar_month=0,
        temp_map=temp_map,
        S_normal=S_normal_v,
        S_bad=S_bad_v,
        ph_kwargs=PH_KWARGS,
        save_dir=path_dir,
        save_full_plans=True,
    )
    all_results[path_name] = result
    wall_path = time.time() - t0_path
    print(f'<<< Done {path_name}: {len(result["roll_logs"])} rolls, '
          f'wall={wall_path/3600:.2f}h')

print(f'\nAll paths complete. Total wall: {(time.time()-t0_global)/3600:.2f}h')


In [ ]:
# ── Load any saved results not already in memory ───────────────────────────
for path_name in PATHS_TO_RUN:
    if path_name not in all_results:
        pkl = os.path.join(SAVE_DIR, path_name, 'result.pkl')
        if os.path.exists(pkl):
            with open(pkl, 'rb') as f:
                all_results[path_name] = pickle.load(f)
            print(f'Loaded {path_name}: {len(all_results[path_name]["roll_logs"])} rolls')
        else:
            print(f'Missing: {path_name} — run the cell above first.')

print(f'\nResults in memory: {list(all_results)}')


In [ ]:
# ── Per-path summary table ──────────────────────────────────────────────────

rows = []
for path_name, result in all_results.items():
    roll_logs = result['roll_logs']
    config    = result['config']
    n_rolls   = len(roll_logs)

    # Biomass at end of horizon (last roll's after-state)
    units_final = roll_logs[-1]['units_df_after'] if roll_logs else None
    npv = compute_npv_from_rolls(roll_logs, units_final, real_horizon=N_ROLLS*3)

    # Per-roll PH stats
    ph_times  = [r.get('ph_total_time', r.get('wallclock_s', 0)) for r in roll_logs]
    ph_iters  = [r.get('ph_n_iters',   r.get('ph_iters', 0))    for r in roll_logs]

    rows.append({
        'Path':           path_name,
        'Rolls':          n_rolls,
        'NPV [MNOK]':     round(npv / 1e6, 0),
        'Avg PH iters':   round(float(np.mean(ph_iters)), 1) if ph_iters else 'n/a',
        'Total wall [h]': round(sum(ph_times) / 3600, 2),
    })

if rows:
    df_summary = pd.DataFrame(rows).set_index('Path')
    print('Rolling SP simulation results (36-month horizon, 27 scenarios/roll):\n')
    display(df_summary)
    print('\nNote: NPV excludes per-month feed costs.')


In [ ]:
# ── Monthly standing biomass timeline per path ──────────────────────────────
# Shows total regional biomass at the start of each committed month.
# Relies on units_df_after from each roll (post-state after 3-month advance).

def total_biomass_kg(units_df):
    """Sum of count × weight_g / 1000 for all non-empty units."""
    total = 0.0
    for _, row in units_df.iterrows():
        c = row.get('count'); w = row.get('avg_weight_g')
        if c is not None and w is not None:
            if not (pd.isna(c) or pd.isna(w)):
                total += float(c) * float(w) / 1000.0
    return total

biomass_timelines = {}
for path_name, result in all_results.items():
    roll_logs = result['roll_logs']
    months, bios = [], []
    for k, log in enumerate(roll_logs):
        t_start = k * 3
        # Beginning-of-roll biomass (before-state)
        bio = total_biomass_kg(log['units_df_before'])
        months.append(t_start); bios.append(bio / 1e6)  # tonnes → ktonnes
    # Append final state
    if roll_logs:
        months.append(N_ROLLS * 3)
        bios.append(total_biomass_kg(roll_logs[-1]['units_df_after']) / 1e6)
    biomass_timelines[path_name] = (months, bios)

# Print as a DataFrame
timeline_df = pd.DataFrame({
    'Month': biomass_timelines[list(biomass_timelines)[0]][0]
})
for name, (months, bios) in biomass_timelines.items():
    timeline_df[name] = bios

print('Standing biomass [thousand tonnes] at roll start:\n')
display(timeline_df.set_index('Month').round(3))

# MAB cap context
print(f'\nRegional MAB cap: {regional_mab/1e6:.0f} ktonnes')
print('Note: biomass is measured in live-weight kg; MAB limit applies to standing biomass.')


In [ ]:
# ── Per-roll PH convergence diagnostics ────────────────────────────────────

for path_name, result in all_results.items():
    roll_logs = result['roll_logs']
    print(f'\n{path_name}:')
    print(f'  {"Roll":>4}  {"Prefix":>8}  {"PH iters":>9}  {"Wall [min]":>10}  {"Eval obj [MNOK]":>15}')
    for log in roll_logs:
        k       = int(log.get('roll_idx', 0))
        prefix  = log.get('prefix_label', '?')
        iters   = int(log.get('ph_n_iters',  log.get('ph_iters',  0)))
        wall_s  = float(log.get('ph_total_time', log.get('wallclock_s', 0)))
        obj     = float(log.get('ph_eval_obj', 0.0))
        print(f'  {k:>4}  {prefix:>8}  {iters:>9}  {wall_s/60:>10.1f}  {obj/1e6:>15.1f}')
